01 - pdf parsing & OCR pipeline 

In [6]:
# CONFIG CELL - Execution parameters and directory paths
import os
import sys
from pathlib import Path

# Dynamically locate project root
CWD = Path(os.getcwd()).resolve()
if (CWD / "data").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "socrr" / "data").exists():
    PROJECT_ROOT = CWD / "socrr"
else:
    PROJECT_ROOT = CWD

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_LABELED_DIR = PROJECT_ROOT / "data" / "labeled"
INTERNAL_CONTROLS_DIR = PROJECT_ROOT / "data" / "internal_controls"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for d in [DATA_INTERIM_DIR, DATA_LABELED_DIR, INTERNAL_CONTROLS_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Project Root: {PROJECT_ROOT}")
print(f"[CONFIG] Raw Data:     {DATA_RAW_DIR}")
print(f"[CONFIG] Interim Data: {DATA_INTERIM_DIR}")

# Pipeline specific configurations
OCR_CHAR_THRESHOLD = 50       # Pages with fewer characters than this are treated as scanned
MAX_TABLE_PAGES_PER_DOC = 150 # Table extraction page limit safeguard
ENABLE_OCR_FALLBACK = True    # Fall back to Tesseract OCR if character count is below threshold

[CONFIG] Project Root: D:\System and Organization controls 2 report info extraction
[CONFIG] Raw Data:     D:\System and Organization controls 2 report info extraction\data\raw
[CONFIG] Interim Data: D:\System and Organization controls 2 report info extraction\data\interim


step 1 - using pymupdf for fast text extraction and pdfplumber for table boundary detection and pytesseract used in OCR when the page is an image or scanned document 

In [7]:
import io
import json
import logging
import shutil
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd
import pymupdf
import pdfplumber
import pytesseract
from PIL import Image

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("SOCRR_Parsing")

# Auto-detect Tesseract executable on Windows or Linux
tesseract_found = False
candidate_paths = [
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
    os.path.expanduser(r"~\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"),
    shutil.which("tesseract")
]

for p in candidate_paths:
    if p and os.path.exists(p):
        pytesseract.pytesseract.tesseract_cmd = p
        tesseract_found = True
        logger.info(f"Tesseract OCR found at: {p}")
        break

if not tesseract_found:
    logger.warning("Tesseract binary not found in standard paths. If scanned pages are encountered, OCR will log a warning and return empty fallback.")

2026-09-29 14:47:58,550 [WARNING] Tesseract binary not found in standard paths. If scanned pages are encountered, OCR will log a warning and return empty fallback.


step 2 - defining extrction functions

1. extract_page_pymupdf- extracts native text with UTF-8 normalization
2. extract_page_ocr- renders page pixmap and runs Tesseract OCR
3. extract_tables_with_pdfplumber- extracts tabular data with clean boundary handling
4. process_single_pdf- combines page parsing and table extaction for a single PDF.

In [8]:
def extract_page_pymupdf(page: pymupdf.Page) -> str:
    """Extract text from a PyMuPDF page object, handling rotation and cleaning."""
    try:
        text = page.get_text("text")
        return text.strip() if text else ""
    except Exception as e:
        logger.warning(f"Error extracting text via PyMuPDF on page {page.number}: {e}")
        return ""

def extract_page_ocr(page: pymupdf.Page) -> str:
    """Render page to pixmap and apply Tesseract OCR fallback."""
    if not tesseract_found:
        logger.warning(f"Page {page.number + 1} appears scanned, but Tesseract binary is not installed.")
        return ""
    try:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_bytes))
        ocr_text = pytesseract.image_to_string(img)
        return ocr_text.strip()
    except Exception as e:
        logger.warning(f"OCR failed on page {page.number + 1}: {e}")
        return ""

def extract_tables_with_pdfplumber(pdf_path: Path, max_pages: Optional[int] = None) -> List[Dict[str, Any]]:
    """Extract all tables from PDF using pdfplumber with defensive error handling."""
    extracted_tables = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            pages_to_process = min(total_pages, max_pages) if max_pages else total_pages
            
            for idx in range(pages_to_process):
                page = pdf.pages[idx]
                try:
                    tables = page.extract_tables()
                    for t_idx, tbl in enumerate(tables):
                        cleaned_rows = []
                        for row in tbl:
                            if row and any(cell and cell.strip() for cell in row if isinstance(cell, str)):
                                cleaned_rows.append([cell.strip() if isinstance(cell, str) else "" for cell in row])
                        if cleaned_rows:
                            extracted_tables.append({
                                "page_num": idx + 1,
                                "table_idx": t_idx + 1,
                                "rows": cleaned_rows
                            })
                except Exception as p_err:
                    logger.debug(f"pdfplumber failed on page {idx+1} of {pdf_path.name}: {p_err}")
    except Exception as e:
        logger.warning(f"Failed to open {pdf_path.name} with pdfplumber: {e}")
    return extracted_tables

def process_single_pdf(pdf_path: Path, char_threshold: int = OCR_CHAR_THRESHOLD) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Process a single PDF file, performing native text extraction, OCR fallback, and table capture."""
    logger.info(f"Processing PDF: {pdf_path.name}")
    doc = pymupdf.open(pdf_path)
    pages_data = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = extract_page_pymupdf(page)
        method = "pymupdf"
        is_scanned = False
        
        if len(text) < char_threshold and ENABLE_OCR_FALLBACK:
            is_scanned = True
            ocr_text = extract_page_ocr(page)
            if len(ocr_text) > len(text):
                text = ocr_text
                method = "tesseract_ocr"
                
        pages_data.append({
            "page_num": page_num + 1,
            "text": text,
            "char_count": len(text),
            "method": method,
            "is_scanned": is_scanned
        })
    doc.close()
    
    tables_data = extract_tables_with_pdfplumber(pdf_path, max_pages=MAX_TABLE_PAGES_PER_DOC)
    logger.info(f"Extracted {len(pages_data)} pages and {len(tables_data)} tables from {pdf_path.name}")
    return pages_data, tables_data

step 3 - running the batch extraction on raw files

iterating over pdf file and save the output to DATA_INTERMIN_DIR

In [9]:
raw_pdfs = list(DATA_RAW_DIR.glob("*.pdf"))
print(f"Found {len(raw_pdfs)} PDF files in {DATA_RAW_DIR}:")
for p in raw_pdfs:
    print(f" - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

summary_records = []

for pdf_file in raw_pdfs:
    report_id = pdf_file.stem
    pages, tables = process_single_pdf(pdf_file)
    
    pages_out_path = DATA_INTERIM_DIR / f"{report_id}_pages.json"
    with open(pages_out_path, "w", encoding="utf-8") as f:
        json.dump(pages, f, indent=2, ensure_ascii=False)
        
    tables_out_path = DATA_INTERIM_DIR / f"{report_id}_tables.json"
    with open(tables_out_path, "w", encoding="utf-8") as f:
        json.dump(tables, f, indent=2, ensure_ascii=False)
        
    scanned_count = sum(1 for p in pages if p["is_scanned"])
    total_chars = sum(p["char_count"] for p in pages)
    
    summary_records.append({
        "report_id": report_id,
        "total_pages": len(pages),
        "total_tables": len(tables),
        "scanned_pages": scanned_count,
        "total_chars": total_chars,
        "pages_file": str(pages_out_path.name),
        "tables_file": str(tables_out_path.name)
    })

summary_file = DATA_INTERIM_DIR / "parsing_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary_records, f, indent=2)

df_summary = pd.DataFrame(summary_records)
display(df_summary)

2026-09-29 14:47:58,589 [INFO] Processing PDF: sample_soc_report.pdf
2026-09-29 14:47:58,626 [WARNING] Page 7 appears scanned, but Tesseract binary is not installed.


Found 3 PDF files in D:\System and Organization controls 2 report info extraction\data\raw:
 - sample_soc_report.pdf (1428.8 KB)
 - synthetic_acme_soc2.pdf (14.1 KB)
 - synthetic_bridge_letter.pdf (2.8 KB)


2026-09-29 14:48:09,282 [INFO] Extracted 118 pages and 123 tables from sample_soc_report.pdf
2026-09-29 14:48:09,294 [INFO] Processing PDF: synthetic_acme_soc2.pdf
2026-09-29 14:48:09,537 [INFO] Extracted 8 pages and 5 tables from synthetic_acme_soc2.pdf
2026-09-29 14:48:09,539 [INFO] Processing PDF: synthetic_bridge_letter.pdf
2026-09-29 14:48:09,584 [INFO] Extracted 1 pages and 0 tables from synthetic_bridge_letter.pdf


,report_id,total_pages,total_tables,scanned_pages,total_chars,pages_file,tables_file
0,sample_soc_report,118,123,1,310692,sample_soc_report_pages.json,sample_soc_report_tables.json
1,synthetic_acme_soc2,8,5,0,10849,synthetic_acme_soc2_pages.json,synthetic_acme_soc2_tables.json
2,synthetic_bridge_letter,1,0,0,1517,synthetic_bridge_letter_pages.json,synthetic_bridge_letter_tables.json


step 4 - just checking

In [10]:
assert len(summary_records) > 0, "No PDF reports were processed!"
for rec in summary_records:
    assert rec["total_pages"] > 0, f"Document {rec['report_id']} produced 0 pages!"
    assert (DATA_INTERIM_DIR / rec["pages_file"]).exists(), f"Pages file {rec['pages_file']} missing!"
    assert (DATA_INTERIM_DIR / rec["tables_file"]).exists(), f"Tables file {rec['tables_file']} missing!"

print("Sanity Check Passed: All raw PDFs parsed and serialized to interim storage.")

Sanity Check Passed: All raw PDFs parsed and serialized to interim storage.
